In [137]:
# ===============================
# 1. 라이브러리
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.metrics import precision_recall_fscore_support

from lightgbm import LGBMClassifier

In [138]:
# ===============================
# 2. 데이터 로드
# ===============================
df = pd.read_csv("../data/insurance_policyholder_churn_synthetic.csv")

In [139]:
# ===============================
# 3. 데이터 전처리
# ===============================
def preprocess_data(df):

    y = df['churn_flag']

    drop_cols = [
        'customer_id',
        'as_of_date',
        'churn_flag',
        'churn_type',
        'churn_probability_true'
    ]

    X = df.drop(columns=[c for c in drop_cols if c in df.columns])

    return X, y


X, y = preprocess_data(df)

In [140]:
# ===============================
# 4. train test split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [141]:
# ===============================
# 5. 컬럼 구분
# ===============================
numeric_features = X.select_dtypes(include=['int64','float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

In [142]:
# ===============================
# 6. 전처리
# ===============================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [143]:
# ===============================
# 7. LightGBM 기본 모델 파이프라인
# ===============================
base_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LGBMClassifier(
        random_state=42,
        class_weight="balanced"
    ))
])


In [136]:
# ===============================
# 8. 하이퍼파라미터 탐색
# ===============================
param_dist = {
    "clf__n_estimators": [200, 300, 500, 700],
    "clf__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "clf__max_depth": [3, 5, 7, 10, -1],
    "clf__num_leaves": [15, 31, 63, 127],
    "clf__subsample": [0.7, 0.8, 0.9, 1.0],
    "clf__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "clf__min_child_samples": [10, 20, 30, 50]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring="recall",   # churn면 recall 중요
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("튜닝 시작...")
search.fit(X_train, y_train)

print("\n===== 튜닝 결과 =====")
print("Best Params:", search.best_params_)
print("Best CV Recall:", search.best_score_)

튜닝 시작...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
[LightGBM] [Info] Number of positive: 12066, number of negative: 27934
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020410 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2747
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 53
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

In [144]:
# ===============================
# 9. 최적 모델
# ===============================
best_model = search.best_estimator_


In [145]:
# ===============================
# 10. 기본 성능 평가 (threshold=0.5)
# ===============================
y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("\n===== 기본 성능 (threshold=0.50) =====")
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1:", f1_score(y_test, y_pred))



===== 기본 성능 (threshold=0.50) =====
              precision    recall  f1-score   support

           0       0.85      0.72      0.78      6983
           1       0.52      0.70      0.60      3017

    accuracy                           0.71     10000
   macro avg       0.68      0.71      0.69     10000
weighted avg       0.75      0.71      0.72     10000

ROC AUC: 0.7851773028403514
F1: 0.5956183745583039


C:\Users\Playdata\miniconda3\envs\tp_returns\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [146]:
# ===============================
# 11. threshold 탐색
# ===============================
print("\n===== Threshold Search =====")

best_threshold = 0.5
best_score = 0

threshold_results = []

for t in np.arange(0.20, 0.71, 0.05):
    y_pred_t = (y_prob >= t).astype(int)

    p, r, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred_t,
        average=None
    )

    precision_1 = p[1]
    recall_1 = r[1]
    f1_1 = f1[1]

    threshold_results.append([t, precision_1, recall_1, f1_1])

    print(f"threshold={t:.2f} | precision1={precision_1:.2f} recall1={recall_1:.2f} f1={f1_1:.2f}")

    # recall을 우선하면서 f1도 너무 낮지 않게
    if recall_1 >= 0.70 and f1_1 > best_score:
        best_score = f1_1
        best_threshold = t

print("\n선택된 threshold:", best_threshold)


===== Threshold Search =====
threshold=0.20 | precision1=0.33 recall1=0.98 f1=0.49
threshold=0.25 | precision1=0.34 recall1=0.97 f1=0.51
threshold=0.30 | precision1=0.36 recall1=0.95 f1=0.52
threshold=0.35 | precision1=0.39 recall1=0.92 f1=0.55
threshold=0.40 | precision1=0.43 recall1=0.86 f1=0.57
threshold=0.45 | precision1=0.47 recall1=0.79 f1=0.59
threshold=0.50 | precision1=0.52 recall1=0.70 f1=0.60
threshold=0.55 | precision1=0.56 recall1=0.61 f1=0.58
threshold=0.60 | precision1=0.63 recall1=0.51 f1=0.57
threshold=0.65 | precision1=0.71 recall1=0.40 f1=0.51
threshold=0.70 | precision1=0.79 recall1=0.27 f1=0.40

선택된 threshold: 0.44999999999999996


In [147]:
# ===============================
# 12. 최종 성능 평가
# ===============================
final_pred = (y_prob >= best_threshold).astype(int)

print("\n===== 최종 성능 =====")
print(classification_report(y_test, final_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1:", f1_score(y_test, final_pred))


===== 최종 성능 =====
              precision    recall  f1-score   support

           0       0.87      0.61      0.72      6983
           1       0.47      0.79      0.59      3017

    accuracy                           0.66     10000
   macro avg       0.67      0.70      0.65     10000
weighted avg       0.75      0.66      0.68     10000

ROC AUC: 0.7851773028403514
F1: 0.5885384709780341


In [148]:
# ===============================
# 13. threshold 결과 표
# ===============================
threshold_df = pd.DataFrame(
    threshold_results,
    columns=["threshold", "precision_1", "recall_1", "f1_1"]
)

print("\n===== Threshold 결과표 =====")
print(threshold_df)


===== Threshold Search =====
threshold=0.30 | precision1=0.36 recall1=0.95 f1=0.52
threshold=0.35 | precision1=0.39 recall1=0.92 f1=0.55
threshold=0.40 | precision1=0.43 recall1=0.86 f1=0.57
threshold=0.45 | precision1=0.47 recall1=0.79 f1=0.59
threshold=0.50 | precision1=0.52 recall1=0.70 f1=0.60
threshold=0.55 | precision1=0.56 recall1=0.61 f1=0.58
threshold=0.60 | precision1=0.63 recall1=0.51 f1=0.57
threshold=0.65 | precision1=0.71 recall1=0.40 f1=0.51
threshold=0.70 | precision1=0.79 recall1=0.27 f1=0.40

Best threshold: 0.3
Best recall: 0.9509446470003314
